# 05 · Temperature texts: Qwen writes, then reads, factual articles and stories

The task: texts that hallucinate (they suddenly change topic mid-sentence, or add wrong facts such as "the capital
of France is London"), made by raising the temperature of the model that writes them. Notebook 06 then checks
whether the graph that evolves while a text is read can tell creative writing from hallucinating.

For each request (3 encyclopedia articles, 3 stories) Qwen writes three versions:

| version | how it is made |
|---|---|
| **cool** (`*_cool`) | temperature `COOL_TEMPERATURE` (0.7) all the way |
| **switch** (`*_switch`) | the cool text up to a random point (25–75% of its tokens), continued at `HOT_TEMPERATURE` |
| **hot** (`*_hot`) | `HOT_TEMPERATURE` all the way |

All versions also use `MIN_P`: tokens below 10% of the #1 token's probability are never drawn. Without it, Qwen
breaks into mixed scripts above temperature ~1.15 (see notebook 03); with it, a hot Qwen is *wrong* but still writes
sentences. Cool and hot use the same `MIN_P`, so temperature is the only thing that changes at the switch.

Everything is kept apart from the 25 real texts (`texts_hallucinated/temperature/`, `results/run_temperature/`).
Then Qwen reads every text exactly as in notebook 01.

**Runtime:** loading ~30 s; each text ~1 minute to write and 3–5 minutes to read. Finished steps are skipped.

In [1]:
import os
import sys
import time
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")   # the model is already in the Hugging Face cache
warnings.filterwarnings("ignore")
# make this folder's qwen_config.py and analysis/ importable, wherever Jupyter was started (e.g. the repo root)
QWEN7B_DIR = Path.cwd()
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src/note_books/Qwen7B/qwen_config.py").exists():
        QWEN7B_DIR = folder / "src/note_books/Qwen7B"
        break
sys.path.insert(0, str(QWEN7B_DIR))

import analysis                                # also makes lca_core (repo root) importable
import numpy as np
import pandas as pd

from analysis.generate import decode, switch_story, write_story
from qwen_config import (COOL_TEMPERATURE, FACT_PROMPTS, GUESSES_PER_WORD, HOT_TEMPERATURE, HOT_TEMPERATURES_TO_PROBE,
                         LOAD_IN_4BIT, MIN_P, MODEL_NAME, RANDOM_SEED, STORY_PROMPTS, TEMPERATURE_DIR,
                         TEMPERATURE_RUN_DIR, TEMPERATURE_TOKENS_DIR, TEMPERATURE_WORDS_DIR, WHOLE_WORD_GUESSES)
from lca_core import LCAMeter
from src.model.load_model import load_model

for folder in [TEMPERATURE_DIR, TEMPERATURE_WORDS_DIR, TEMPERATURE_TOKENS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# every request: (text kind, request key, prompt)
REQUESTS = []
for key, prompt in FACT_PROMPTS.items():
    REQUESTS.append(("factual", key, prompt))
for key, prompt in STORY_PROMPTS.items():
    REQUESTS.append(("creative", key, prompt))

## 1. Load the model

In [2]:
started = time.time()
tokenizer, model = load_model(MODEL_NAME, load_in_4bit=LOAD_IN_4BIT)
meter = LCAMeter(model=model, tokenizer=tokenizer)
print(f"loaded in {time.time() - started:.0f} s")

system check:
Python path: C:\Users\yoav7\miniconda3\python.exe
Torch version: 2.6.0+cu124
GPU: NVIDIA GeForce RTX 4060
PyTorch: 2.6.0+cu124
CUDA: 12.4
CUDA available: True
Capability: (8, 9)
this is include the following functions torch , torch.nn.functional as F, transformers.AutoTokenizer, transformers.AutoModelForCausalLM


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded in 19 s


## 2. Pick the hot temperature
The beginning of the Paris article (250 tokens) at each temperature in `HOT_TEMPERATURES_TO_PROBE`, all with `MIN_P`.
The hot temperature is the highest one whose text is still English sentences but has wrong facts or topic changes;
it is set as `HOT_TEMPERATURE` in `qwen_config.py`. Delete `probe.txt` to write the probes again.

In [3]:
probe_path = TEMPERATURE_DIR / "probe.txt"
if not probe_path.exists():
    parts = []
    for temperature in HOT_TEMPERATURES_TO_PROBE:
        ids = write_story(model, tokenizer, FACT_PROMPTS["paris"], temperature, RANDOM_SEED, max_new_tokens=250,
                          kind="factual", min_p=MIN_P)
        parts.append(f"===== temperature {temperature}, min_p {MIN_P} =====" + chr(10) + decode(tokenizer, ids) + chr(10))
    probe_path.write_text(chr(10).join(parts), encoding="utf-8")
print(probe_path.read_text(encoding="utf-8"))
print(f"chosen: HOT_TEMPERATURE = {HOT_TEMPERATURE}")

===== temperature 1.3, min_p 0.1 =====
Paris, often referred to as the “City of Light,” is the capital and most populous city of France. Located in the northern part of the country along the Seine River, Paris has been a major cultural, economic, and political center for centuries. It is renowned for its historic architecture, artistic masterpieces, and cultural landmarks. The city's history dates back to Roman times, and it has played a crucial role in the development of Western civilization.

The city’s urban layout is famously divided into two main parts by the Seine River: the Left Bank (Rive gauche) and the Right Bank (Rive droite). The central area known as the Île de la Cité, an ancient island in the middle of the Seine, is where Notre-Dame Cathedral and other historical structures are located. The city is famous for its numerous parks, such as the Luxembourg Gardens, the Tuileries Garden, and the expansive Bois de Boulogne and Bois de Vincennes. These green spaces offer residen

## 3. Write the texts
For each request: the cool text, the switch text (cool up to a random switch point, then hot) and the hot text.
The random draws use `RANDOM_SEED`. Texts already written are kept (their hallucinations are marked by hand in
`temperature/marks.csv`); delete `temperature/switch_points.csv` to write them again.

In [4]:
rng = np.random.default_rng(RANDOM_SEED)
switch_rows = []
already_written = (TEMPERATURE_DIR / "switch_points.csv").exists()
for request_number, (kind, key, prompt) in enumerate(REQUESTS):
    if already_written:
        break
    seed = RANDOM_SEED + request_number
    cool_name = f"{kind}_temp_{key}_cool"
    switch_name = f"{kind}_temp_{key}_switch"
    hot_name = f"{kind}_temp_{key}_hot"
    started = time.time()

    cool_ids = write_story(model, tokenizer, prompt, COOL_TEMPERATURE, seed, kind=kind, min_p=MIN_P)
    (TEMPERATURE_DIR / f"{cool_name}.txt").write_text(decode(tokenizer, cool_ids), encoding="utf-8")

    switch_ids, switch_token = switch_story(model, tokenizer, prompt, cool_ids, seed, HOT_TEMPERATURE, rng,
                                            kind=kind, min_p=MIN_P)
    (TEMPERATURE_DIR / f"{switch_name}.txt").write_text(decode(tokenizer, switch_ids), encoding="utf-8")
    switch_rows.append({"text": switch_name, "clean original": cool_name, "switch token": switch_token,
                        "switch char": len(decode(tokenizer, cool_ids[:switch_token])), "story tokens": len(switch_ids)})

    hot_ids = write_story(model, tokenizer, prompt, HOT_TEMPERATURE, seed, kind=kind, min_p=MIN_P)
    (TEMPERATURE_DIR / f"{hot_name}.txt").write_text(decode(tokenizer, hot_ids), encoding="utf-8")
    print(f"{kind:8s} {key:10s} {len(cool_ids)} / {len(switch_ids)} / {len(hot_ids)} tokens, "
          f"switch at token {switch_token}, {time.time() - started:.0f} s", flush=True)

if already_written:
    switch_points = pd.read_csv(TEMPERATURE_DIR / "switch_points.csv")
else:
    switch_points = pd.DataFrame(switch_rows)
    switch_points.to_csv(TEMPERATURE_DIR / "switch_points.csv", index=False)

# a switch text must be its cool twin, word for word, up to the switch point
for _index, switch in switch_points.iterrows():
    switch_text = (TEMPERATURE_DIR / f"{switch['text']}.txt").read_text(encoding="utf-8")
    cool_text = (TEMPERATURE_DIR / f"{switch['clean original']}.txt").read_text(encoding="utf-8")
    char = int(switch["switch char"])
    print(f"{switch['text']:34s} same as its cool twin up to the switch: {switch_text[:char] == cool_text[:char]}")
switch_points

factual  paris      853 / 853 / 849 tokens, switch at token 576, 180 s


factual  apollo     644 / 541 / 1000 tokens, switch at token 366, 154 s


factual  vaccines   739 / 739 / 908 tokens, switch at token 373, 160 s


creative magic      1000 / 1000 / 929 tokens, switch at token 385, 200 s


creative fairy      751 / 751 / 1000 tokens, switch at token 303, 162 s


creative detective  1000 / 759 / 1000 tokens, switch at token 270, 198 s


factual_temp_paris_switch          same as its cool twin up to the switch: True
factual_temp_apollo_switch         same as its cool twin up to the switch: True
factual_temp_vaccines_switch       same as its cool twin up to the switch: True
creative_temp_magic_switch         same as its cool twin up to the switch: True
creative_temp_fairy_switch         same as its cool twin up to the switch: True
creative_temp_detective_switch     same as its cool twin up to the switch: True


,text,clean original,switch token,switch char,story tokens
0,factual_temp_paris_switch,factual_temp_paris_cool,576,2696,853
1,factual_temp_apollo_switch,factual_temp_apollo_cool,366,1712,541
2,factual_temp_vaccines_switch,factual_temp_vaccines_cool,373,1963,739
3,creative_temp_magic_switch,creative_temp_magic_cool,385,1648,1000
4,creative_temp_fairy_switch,creative_temp_fairy_cool,303,1367,751
5,creative_temp_detective_switch,creative_temp_detective_cool,270,1264,759


## 4. Qwen reads every text
Same measurements as notebook 01. Texts with saved results are skipped.

In [5]:
texts_to_read = {}
for path in sorted(TEMPERATURE_DIR.glob("*_temp_*.txt")):
    texts_to_read[path.stem] = path.read_text(encoding="utf-8")

log_path = TEMPERATURE_RUN_DIR / "run_log.csv"
if log_path.exists():
    run_log = pd.read_csv(log_path)
else:
    run_log = pd.DataFrame(columns=["text", "measured words", "tokens", "seconds"])

for name, text in texts_to_read.items():
    words_path = TEMPERATURE_WORDS_DIR / f"{name}.csv"
    tokens_path = TEMPERATURE_TOKENS_DIR / f"{name}.csv"
    if words_path.exists() and tokens_path.exists():
        continue
    started = time.time()
    words = meter.run(text, k=GUESSES_PER_WORD, complete_words=WHOLE_WORD_GUESSES)
    words.to_csv(words_path, index=False)
    meter.last_tokens.to_csv(tokens_path, index=False)
    seconds = round(time.time() - started)
    new_row = {"text": name, "measured words": len(words), "tokens": len(meter.last_tokens), "seconds": seconds}
    run_log = pd.concat([run_log[run_log["text"] != name], pd.DataFrame([new_row])], ignore_index=True)
    run_log.to_csv(log_path, index=False)
    print(f"{name:34s} {len(words):4d} words  {len(meter.last_tokens):5d} tokens  {seconds:5d} s", flush=True)
run_log

creative_temp_detective_cool        239 words   1000 tokens    483 s


creative_temp_detective_hot         311 words    999 tokens    371 s


creative_temp_detective_switch      237 words    759 tokens    278 s


creative_temp_fairy_cool            228 words    751 tokens    307 s


creative_temp_fairy_hot             341 words    998 tokens    507 s


creative_temp_fairy_switch          240 words    750 tokens    340 s


... (9 outputs left out)

factual_temp_vaccines_cool          261 words    739 tokens    298 s


factual_temp_vaccines_hot           279 words    908 tokens    321 s


factual_temp_vaccines_switch        245 words    739 tokens    273 s


,text,measured words,tokens,seconds
0,creative_temp_detective_cool,239,1000,483
1,creative_temp_detective_hot,311,999,371
2,creative_temp_detective_switch,237,759,278
3,creative_temp_fairy_cool,228,751,307
4,creative_temp_fairy_hot,341,998,507
5,creative_temp_fairy_switch,240,750,340
6,creative_temp_magic_cool,256,1000,330
7,creative_temp_magic_hot,245,928,294
8,creative_temp_magic_switch,324,1000,407
9,factual_temp_apollo_cool,171,644,231
